### Lab 01 — Experiments (Part D, F, G, H, I)



In [124]:
import json
import math
import numpy as np
import scipy.sparse as sp
import pandas as pd
from preprocessing import tokenize_minimal, tokenize_normalized, tokenize_extended
from implementation import cosine_similarity  

DATA_PATH = r"D:\\NLP\\lab1\\dataset\\30k_documents.json"


#### Part D — Experiment 1: Inspect the Sparse Representation

In [125]:
texts = []
with open(DATA_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        texts.append(json.loads(line)['text'])

token_per_doc = [d.lower().split() for d in texts]

vocab = set()
df_counts = {}
nnz = 0
for tokens in token_per_doc:
    uniq = set(tokens)
    vocab.update(uniq)
    nnz += len(uniq)
    for w in uniq:
        df_counts[w] = df_counts.get(w, 0) + 1

N = len(texts)
V = len(vocab)
sparsity = 1 - nnz / (N * V)

print(f"Number of documents(N) = {N:,}")
print(f"Vocabulary size(V) = {V:,}")
print(f"Matrix shape R^(N x V) = ({N}, {V})")
print(f"Check Sparsity: S = 1 - nnz(X)/(N*V) = {sparsity:.4f}")

term_fre = []
for tokens in token_per_doc:
    counts = {}
    for w in tokens:
        counts[w] = counts.get(w, 0) + 1
    term_fre.append(counts)

idf = {w: math.log(N / df_counts[w]) for w in vocab}


Number of documents(N) = 30,000
Vocabulary size(V) = 473,388
Matrix shape R^(N x V) = (30000, 473388)
Check Sparsity: S = 1 - nnz(X)/(N*V) = 0.9996


**Tại sao một document chỉ dùng một phần rất nhỏ vocabulary nhưng vector vẫn có chiều V?**

Vì vocabulary V được xây dựng là list của tất cả các từ xuất hiện trong corpus, một document chỉ chứa một phàn nhỏ từ trong vocabulary, nhưng để có thể so sánh các document với nhau, tất cả vector phải nằm chung một không gian toạ độ V — nên phần lớn toạ độ của một document sẽ là 0.


7.5 Inspect vocabulary. Tìm:
- 20 terms phổ biến nhất theo document frequency;
- 20 terms có IDF cao nhất;
- 20 terms TF-IDF cao nhất trong 1 document được chọn

In [126]:
top20_df = sorted(df_counts.items(), key=lambda kv: kv[1], reverse=True)[:20]
print("--- Top 20 most popular terms by document frequency ---")
top20_df


--- Top 20 most popular terms by document frequency ---


[('the', 27870),
 ('and', 27385),
 ('to', 26646),
 ('of', 25999),
 ('a', 25753),
 ('in', 25042),
 ('for', 23556),
 ('is', 22673),
 ('with', 21323),
 ('on', 19936),
 ('that', 18042),
 ('this', 17548),
 ('are', 17487),
 ('as', 16355),
 ('at', 16233),
 ('from', 16183),
 ('be', 16025),
 ('it', 15474),
 ('you', 15361),
 ('by', 14963)]

In [127]:
top20_idf = sorted(idf.items(), key=lambda kv: kv[1], reverse=True)[:20]
top_idf_table = pd.DataFrame([(t, round(v, 5)) for t, v in top20_idf], columns=["Term", "IDF"])
print("--- Top 20 highest IDF terms  ---")
top_idf_table


--- Top 20 highest IDF terms  ---


,Term,IDF
0,wedge-shaped,10.30895
1,skillfully.,10.30895
2,boutonnieres,10.30895
3,(strings),10.30895
4,teede.,10.30895
5,kettle>hot,10.30895
6,allegiences,10.30895
7,slave-girl,10.30895
8,archaeology’s,10.30895
9,colorburst,10.30895


In [128]:
doc_id = 0
print(f"========== Document {doc_id} ==========")

total_doc0 = sum(term_fre[doc_id].values())
tfidf_doc0 = {w: (c / total_doc0) * idf.get(w, 0) for w, c in term_fre[doc_id].items()}
top20_tfidf_doc0 = sorted(tfidf_doc0.items(), key=lambda kv: kv[1], reverse=True)[:20]
top_tfidf_table = pd.DataFrame([(t, round(v, 4)) for t, v in top20_tfidf_doc0], columns=["Term", "TF-IDF"])
print("--- 20 highest term's TF-IDF in document 0 ---")
top_tfidf_table


========== Document 0 ==========
--- 20 highest term's TF-IDF in document 0 ---


,Term,TF-IDF
0,bbq,0.1402
1,class,0.1062
2,missoula!,0.0793
3,bbq?,0.0793
4,balay,0.0793
5,meat,0.0783
6,lonestar,0.0740
7,kcbs,0.0740
8,"timelines,",0.0740
9,rangers.,0.0633


#### Part F — Experiment 2: Preprocessing Ablation

In [129]:
def build_count_matrix(tokenized_docs):
    """Build sparse count matrix"""
    vocab = {}
    rows, cols, data = [], [], []
    for i, tokens in enumerate(tokenized_docs):
        counts = {}
        for tok in tokens:
            counts[tok] = counts.get(tok, 0) + 1
        for tok, c in counts.items():
            j = vocab.get(tok)
            if j is None:
                j = len(vocab)
                vocab[tok] = j
            rows.append(i); cols.append(j); data.append(c)
    X = sp.csr_matrix((data, (rows, cols)), shape=(len(tokenized_docs), len(vocab)))
    return X, vocab


def compute_tfidf_matrix(X):
    """tf(t,d)=c/tong, idf(t)=log(N/df(t)), tfidf=tf*idf"""
    N, V = X.shape
    sums_word = np.asarray(X.sum(axis=1)).ravel()
    sums_word[sums_word == 0] = 1   
    tf = sp.diags(1.0 / sums_word) @ X
    df = X.getnnz(axis=0)   
    idf = np.log(N / df)                          
    tf_idf = tf @ sp.diags(idf)
    return tf, df, idf, tf_idf


def normalize_rows(X):
    norms = np.sqrt(np.asarray(X.multiply(X).sum(axis=1))).ravel()
    norms[norms == 0] = 1
    return sp.diags(1.0 / norms) @ X


In [130]:
def pipeline(func_tokenizer):
    tok = [func_tokenizer(t) for t in texts]
    X, vocab = build_count_matrix(tok)
    tf, df, idf, tf_idf = compute_tfidf_matrix(X)
    tf_idf_normalized = normalize_rows(tf_idf)

    avg_tokens = float(np.mean([len(x) for x in tok]))
    sparsity = 1 - X.nnz / (X.shape[0] * X.shape[1])

    def search_engine(query, top_k=5):
        qtok = func_tokenizer(query)
        qcounts = {}
        for t in qtok:
            qcounts[t] = qcounts.get(t, 0) + 1

        total_toks = len(qtok) or 1
        qvec = np.zeros(len(vocab))

        for t, c in qcounts.items():
            j = vocab.get(t)
            if j is not None:
                qvec[j] = (c / total_toks) * idf[j]
        norm = np.linalg.norm(qvec)
        
        if norm == 0:
            return []
        qvec = qvec / norm
        simulars = tf_idf_normalized @ qvec
        top_idx = np.argsort(-simulars)[:top_k]
        return [(int(i), float(simulars[i])) for i in top_idx]

    # out of vocabulary
    def oov_rate(queries):
        total_toks_tok, oov_tok = 0, 0
        for q in queries:
            qtok = func_tokenizer(q)
            total_toks_tok += len(qtok)
            oov_tok += sum(1 for t in qtok if t not in vocab)
        return oov_tok / total_toks_tok if total_toks_tok else 0.0

    return {"vocab": vocab, "idf": idf, "TF-IDF_norm": tf_idf_normalized,
            "vocab_size": len(vocab), "avg_tokens": avg_tokens, "sparsity": sparsity,
            "oov_rate": oov_rate,"search_engine": search_engine}


 
pipe_A = pipeline(tokenize_minimal)
pipe_B = pipeline(tokenize_normalized)
pipe_C = pipeline(tokenize_extended)


In [131]:
topics = {
    "bbq class": ["bbq class"],
    "credit card": ["credit card"],
    "climate change": ["climate change"],
    "national park": ["national park"],
    "machine learning": ["machine learning"],
    "contact lenses": ["contact lenses"],
    "car insurance": ["car insurance"],
    "heart attack": ["heart attack", "myocardial infarction"],
}

lower_tex = [t.lower() for t in texts]
relevances = {}
for q, phrases in topics.items():
    ids = set(i for i, sen in enumerate(lower_tex) if any(p in sen for p in phrases))
    relevances[q] = ids

pd.DataFrame({"Phrase": list(relevances), "Number of relevant documents": [len(v) for v in relevances.values()]})


,Phrase,Number of relevant documents
0,bbq class,1
1,credit card,265
2,climate change,127
3,national park,112
4,machine learning,48
5,contact lenses,10
6,car insurance,31
7,heart attack,50


In [132]:
def evaluate(search_engine, relevances_dict, k=5):
    rows = []
    for q, rel_id in relevances_dict.items():
        res_search = search_engine(q, top_k=k)
        searched_id = [d_id for d_id,_ in res_search]
        checked_rel_id = [d for d in searched_id if d in rel_id]

        precision_k = len(checked_rel_id) / k
        recal_k = len(checked_rel_id) / len(rel_id) if rel_id else 0.0
        
        # ket qua dau tien xuat hien o vi tri thu may
        RR = 0.0
        for rank, d in enumerate(searched_id, start=1):
            if d in rel_id:
                RR = 1.0 / rank
                break
        rows.append({"query": q, "num_relevant_docs": len(rel_id), "P@5": precision_k, "R@5": recal_k, "Reciprocal Rank": RR})
    return pd.DataFrame(rows)


query_list = list(topics.keys())
ablation_rows = []
for name, pipe in [("Pipeline A", pipe_A), ("Pipeline B", pipe_B), ("Pipeline C", pipe_C)]:
    eval = evaluate(pipe["search_engine"], relevances)
    ablation_rows.append({
        "Pipeline": name,
        "Vocabulary size": pipe["vocab_size"],
        "Average tokens/document": round(pipe["avg_tokens"], 2),
        "Matrix sparsity": round(pipe["sparsity"], 4),
        "OOV rate": round(pipe["oov_rate"](query_list), 4),
        "Search performance": round(eval["P@5"].mean(), 4),
    })

ablation_df = pd.DataFrame(ablation_rows).set_index("Pipeline").T
ablation_df


Pipeline,Pipeline A,Pipeline B,Pipeline C
Vocabulary size,473388.0000,186465.0000,168457.0000
Average tokens/document,361.0900,209.4600,434.4500
Matrix sparsity,0.9996,0.9993,0.9989
OOV rate,0.0000,0.0000,0.0000
Search performance,0.5750,0.5000,0.4500


#### 9.5 Câu hỏi phân tích

1. Lowercasing làm thay đổi vocabulary như thế nào?

Lowercasing được áp dụng ở cả 3 pipeline và có tác dụng làm cho các biến thể viết hoa/thường của cùng 1 từ (`The` / `the` / `THE`) - những từ đáng ra sẽ bị tách thành các entry khác nhau trong vocabulary, làm vocabulary phình to ra; quy về thành cùng 1 định dạng viết thường, được lưu trong vocabulary làm cho vocab có tính unique và có kích thước nhỏ hơn. 


2. Stopword removal có luôn cải thiện representation không? 

Không, so sánh 2 Pipeline A (không lọc stopword) và Pipeline B (có lọc): vocabulary size thay đổi đáng kể(473,388 → 186,465), kéo theo Average tokens/document cũng giảm (361.09 → 209.46) và search performance cũng bị giảmd độ chính xác đi. Do đó, mặc dù stopword removal giúp nén index gọn hơn (ít token phải lưu/tính TF hơn) mà không làm giảm chất lượng tìm kiếm trong thí nghiệm này — nhưng cũng không chưa đảm bảo được về mặt chất lượng truy vấn, vì stopword vốn đã có IDF thấp nên gần như không đóng góp vào TF-IDF/cosine similarity dù có bị loại hay không.

3. Việc loại punctuation có thể làm mất thông tin gì? 

Loại dấu câu có thể làm mất ranh giới câu/cụm từ, gộp nhầm các con số/ký hiệu có nghĩa (ví dụ giá tiền "$35", email, hoặc "U.S." bị tách thành "u" và "s"), và làm mất các emoticon/ký hiệu mang sắc thái (quan trọng với bài toán sentiment, tuy không ảnh hưởng nhiều tới bài toán search theo từ khoá ở đây).

4. Pipeline nào tạo ra sparse matrix nhất? 

Pipeline A có sparsity cao nhất (0.9996).

5. Pipeline nào cho search tốt nhất?

Pipeline A và B có search performance tương đương nhau lần lượt là 0.5750 và 0.5000, Pipeline C kém nhất với search performance là 0.45. Mặc dù Pipeline B đạt chất lượng tìm kiếm có thể nói là tương đương với Pipeline A nhưng index ít hơn nên có thể nói Pipeline B cho chất lượng search tốt nhất.

6. Search tốt hơn có đồng nghĩa với vocabulary nhỏ hơn không?

Không — Ngược lại với nhận định đó, pipeline A là pipeline có vocab size lớn nhất nhưng lại cho kết quả tìm kiếm với độ chính xác lớn nhất trong khi pipeline C có vocab size nhỏ hơn rất nhiều nhưng độ chính xác khi tìm kiếm cũng nhỏ hơn rất nhiều. Nên chưa thể khẳng định search tốt hơn đồng nghĩa với vocabulary nhỏ hơn.

### Part G — Application: Build a Document Search Engine

In [133]:
MAIN_PROCESSING = pipe_B

def preview(i, n=100):
    return texts[i][:n].replace("\n", " ") + "..."

def run_query_table(query, top_k=5):
    res = MAIN_PROCESSING["search_engine"](query, top_k=top_k)
    rows = [{"Rank": r, "Document ID": doc_id, "Similarity": round(sim, 4), "Document preview": preview(doc_id)}
            for r, (doc_id, sim) in enumerate(res, start=1)]
    return pd.DataFrame(rows)

example_queries = [
    "medical image classification",
    "transformer language model",
    "deep learning healthcare",
    "natural language processing"
]

for q in example_queries:
    print("="*80)
    print("query:", q)
    display(run_query_table(q))


query: medical image classification


,Rank,Document ID,Similarity,Document preview
0,1,18971,0.4465,The new RTS Environmental Classification syste...
1,2,8527,0.3678,History of maize classification. How races use...
2,3,19908,0.2351,Download League Of Legends Wallpapers in high-...
3,4,15682,0.2323,- Group Image: Provided functionality of group...
4,5,17794,0.2305,Filters the output of 'wp_calculate_image_size...


query: transformer language model


,Rank,Document ID,Similarity,Document preview
0,1,27936,0.5079,"hi, I am having problems with transformer / ci..."
1,2,25428,0.2722,"Note: If you're on an iPhone, you cannot chang..."
2,3,24482,0.2169,"Harald, you are a co-owner of Language Partner..."
3,4,701,0.2036,Program in Teaching French as a Foreign Langua...
4,5,4289,0.2000,"Commercial Building Properties, Commercial Bui..."


query: deep learning healthcare


,Rank,Document ID,Similarity,Document preview
0,1,11119,0.3109,The opportunities offered by Big Data will onl...
1,2,6123,0.3095,"With today’s advancement in technology, it is ..."
2,3,11979,0.3092,Doctorate of Healthcare Organization Program i...
3,4,9252,0.2993,"SAN DIEGO AND WASHINGTON, D.C. – Sept. 5, 2018..."
4,5,11777,0.2560,It is much cheaper (50–70% cheaper) to fly to ...


query: natural language processing


,Rank,Document ID,Similarity,Document preview
0,1,25428,0.3808,"Note: If you're on an iPhone, you cannot chang..."
1,2,8705,0.3734,These regulations may be called the Food Safet...
2,3,24482,0.3033,"Harald, you are a co-owner of Language Partner..."
3,4,701,0.2847,Program in Teaching French as a Foreign Langua...
4,5,4075,0.2779,Looking for Spanish language instructor to imp...


## Part H — Evaluation

In [134]:
my_queries = [
    "real estate",
    "insurance policy",
    "swimming pool",
    "online shopping",
    "solar panels",
    "tax return",
    "coffee shop",
    "health insurance plans",
    "search engine optimization",
    "customer service representative",
]

my_relevances = {}
for q in my_queries:
    p = q.lower()
    ids = set(i for i, sen in enumerate(lower_tex) if p in sen)
    my_relevances[q] = ids

eval_my = evaluate(MAIN_PROCESSING["search_engine"], my_relevances, k=5)
print("Mean P@5:", round(eval_my["P@5"].mean(), 4))
print("Mean R@5:", round(eval_my["R@5"].mean(), 4))
print("Mean RR :", round(eval_my["Reciprocal Rank"].mean(), 4))
eval_my


Mean P@5: 0.64
Mean R@5: 0.0818
Mean RR : 0.795


,query,num_relevant_docs,P@5,R@5,Reciprocal Rank
0,real estate,372,1.0,0.013441,1.00
1,insurance policy,59,0.8,0.067797,1.00
2,swimming pool,105,1.0,0.047619,1.00
3,online shopping,43,0.2,0.023256,0.20
4,solar panels,40,1.0,0.125000,1.00
5,tax return,39,0.8,0.102564,1.00
6,coffee shop,47,0.2,0.021277,0.50
7,health insurance plans,8,0.4,0.250000,1.00
8,search engine optimization,37,0.8,0.108108,1.00
9,customer service representative,17,0.2,0.058824,0.25


## Part I — Error Analysis

Chọn theo kết quả bảng Part H: **2 query tốt** = `climate change` (P@5=1.0), `credit card` (P@5=0.8). **2 query kém** = `machine learning` (P@5=0.2), `contact lenses` (P@5=0.2).


In [135]:
def error_case(query, relevances_dict, top_k=5):
    rel = relevances_dict[query]
    res = MAIN_PROCESSING["search_engine"](query, top_k=top_k)
    print("Query:", query)
    print("Number of relevant documents:", len(rel))
    sample_rel = sorted(rel)[:10]
    print(f"Expected relevant documents (sample {len(sample_rel)}/{len(rel)}):", sample_rel)
    print("Retrieved documents:")
    for rank, (doc_id, sim) in enumerate(res, start=1):
        tag = "RELEVANT" if doc_id in rel else "Don't relevant"
        print(f"  {rank}. doc {doc_id}  sim={sim:.4f}  [{tag}]  {preview(doc_id, 90)}")
    print()

for q in ["real estate", "solar panels", "online shopping", "customer service representative"]:
    error_case(q, my_relevances)


Query: real estate
Number of relevant documents: 372
Expected relevant documents (sample 10/372): [17, 61, 356, 454, 517, 637, 710, 714, 741, 885]
Retrieved documents:
  1. doc 19519  sim=0.6034  [RELEVANT]  Buying or Selling Real Estate call Ria today! Hello, I'm Ria Masterson, and I began my rea...
  2. doc 27406  sim=0.5467  [RELEVANT]  We understand that buying or selling a home is more than just a transaction: it’s a life-c...
  3. doc 14874  sim=0.5222  [RELEVANT]  Nov. 15, 2017 11:55 a.m. Real estate agents will no longer be able to act on behalf of bot...
  4. doc 11477  sim=0.4952  [RELEVANT]  Protect Your Assets and Your Family With an Estate Plan – Contrary to what you might think...
  5. doc 1616  sim=0.4839  [RELEVANT]  With its beautiful beaches, it's no wonder so many tourists flock to Long Beach Island eve...

Query: solar panels
Number of relevant documents: 40
Expected relevant documents (sample 10/40): [471, 1593, 3450, 3545, 4679, 5179, 7541, 7687, 7885, 8524]
Retri

**Real Estate**

1. Vì sao document đứng đầu (rank 1) được xếp hạng cao nhất? Document đứng đầu rank vì nó có chỉ số tương đồng(sim) cao nhất so với các kết quả tìm được. Khi đi sâu vào chi tiết văn bản có thể thấy cụm truy vấn xuất hiện 7 lần trên toàn bộ document và điều này rõ ràng cho thấy tần xuất suất hiện của cụm này khá laf dày nên các chỉ số tf, tf-idf cũng cao.

2. Những từ nào trong query đóng góp nhiều vào similarity?  Do query chỉ có 2 từ và tạo thành 1 cụm cố định nên cả 2 từ đều góp phần tạo neen giá trị thu được của sim

3. Có lexical overlap giữa query và các document relevant không? Có - nhìn vào document review thì cũng có thể thấy cụm Real Estate xuất hiện đều ở cả 5 kết quả thu được.

4. Có relevant document nào bị bỏ sót không? Vì sao? Không có relevant document nào bị bỏ sót vì việc searching dựa trên việc chữ tìm kiếm có xuất hiện trong đoan vặn hay không nên có thể dự đoán là khả năng bị bỏ sót là thấp.

5. Failure (nếu có) xuất phát từ đâu: preprocessing, TF, IDF, vocabulary, lexical matching, hay nguyên nhân khác?

**online shopping**

1. Vì sao document đứng đầu (rank 1) được xếp hạng cao nhất? Document đứng đầu rank vì nó có chỉ số tương đồng(sim) cao nhất so với các kết quả tìm được. Thế nhưng đây không phải là văn bản liên quan đến query do văn bản chỉ nói đến "Shopping" là 1 trong số 2 từ của cụm truy vấn nên sự lặp lại nhiều của từ đã khiến cho chỉ số tương đồng của văn bản này đối với cụm truy vấn là cao.

2. Những từ nào trong query đóng góp nhiều vào similarity? Từ "Shopping" là từ được nhắc đến nhiều cũng như là ảnh hưởng lớn vào similarity. Điều này có thể được thấy ở văn bản 29898 có 4 câu nhưng 3/4 câu có nhắc tới chữ shopping.

3. Có lexical overlap giữa query và các document relevant không? Có nhưng tần suất không nhiều

4. Có relevant document nào bị bỏ sót không? Vì sao? Không có relevant document nào bị bỏ sót vì việc searching dựa trên việc chữ tìm kiếm có xuất hiện trong đoan vặn hay không nên có thể dự đoán là khả năng bị bỏ sót là thấp.

5. Failure (nếu có) xuất phát từ đâu: preprocessing, TF, IDF, vocabulary, lexical matching, hay nguyên nhân khác? Từ lexical matching / bag-of-words: TF-IDF đếm từng từ riêng lẻ ("shopping") mà không quan tâm nó có đứng cạnh từ "online" để tạo thành đúng khái niệm "mua sắm trực tuyến" hay không. Document nói về shopping nói chung (cửa hàng vật lý, tên riêng "Swap Shop") vẫn được tính similarity cao chỉ vì trùng 1 từ — đây là hạn chế giống case "heart attack".


## Part J — From Failure to the Next NLP Representation

Thất bại "heart attack ≠ myocardial infarction" cho thấy: để hệ thống nhận ra 2 cụm từ có **nghĩa giống nhau** dù không chia sẻ token nào, ta cần một representation trong đó các từ/cụm từ xuất hiện trong ngữ cảnh tương tự được biểu diễn bằng **vector gần nhau** — tức là **distributional representation / word embedding**, thay vì đếm từ (TF-IDF) như hiện tại.

```
TF-IDF -> Distributional representation -> Word Embedding -> Contextual Embedding -> Transformer
```
